<a href="https://colab.research.google.com/github/chiraswykj/GENAI_TUTORIALS/blob/main/Intelligent_Document_Chatbot_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Mon Sep  7 06:43:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu pypdf gradio langchain langchain-community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

from sentence_transformers import SentenceTransformer

from pypdf import PdfReader

import faiss
import gradio as gr

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [ ]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("Model:", model_name)

Model: Qwen/Qwen2.5-0.5B-Instruct


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully!


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("LLM loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully!


In [ ]:
print("Model device:", model.device)

Model device: cuda:0


In [ ]:
messages = [
    {
        "role": "user",
        "content": "Explain what an LLM is in very simple words."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

An LLM stands for Large Language Model. It's like a big computer program that can understand and generate human language, much like how you use your brain to think and communicate with others. These programs learn from a lot of data to become better at understanding the world around them and generating useful responses.


In [ ]:
def ask_llm(question):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return answer

In [ ]:
answer = ask_llm("What is machine learning?")

print(answer)

Machine Learning is an interdisciplinary field that focuses on the development of algorithms and statistical models that enable computers to learn from and make predictions or decisions without being explicitly programmed. It involves training systems with data to improve their performance in tasks such as image recognition, speech recognition, natural language processing, and more.

Key components of Machine Learning include:

1. **Data**: This is the raw information used to train the model.
2. **Models**: These are mathematical functions or algorithms that can be trained to recognize patterns in the data.
3. **Training**: The process where the system learns how to use these models to perform specific tasks.
4. **Evaluation**: Measures the performance of the learned model using real-world data to assess its effectiveness.

Machine Learning has become


In [ ]:
print(ask_llm("What is artificial intelligence?"))

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and act like humans. AI encompasses a wide range of technologies, including machine learning, natural language processing, computer vision, robotics, autonomous systems, and others.

AI can be broadly divided into two main categories: rule-based AI and deep learning AI. Rule-based AI relies on pre-programmed rules and algorithms to make decisions, while deep learning AI uses neural networks to learn from data and improve its performance over time.

AI has several applications across various industries and sectors. It has been used in areas such as healthcare, finance, transportation, manufacturing, and more. However, there are also concerns about the ethical implications of AI, particularly around issues like bias


In [ ]:
print(ask_llm("What is Python?"))

Python is an interpreted high-level programming language developed by Guido van Rossum and first released in 1991. It has gained significant popularity due to its simplicity, readability, versatility, and ease of use.

Key features of Python include:

1. **Interpreted Language**: Unlike compiled languages like C or C++, Python executes code directly on the computer's processor without compiling it into machine code first.
2. **Dynamic Typing**: Variables can be assigned any type at runtime, which makes it easy to write programs that work with different data types.
3. **Object-Oriented Programming (OOP)**: Python supports OOP concepts such as classes, objects, inheritance, and polymorphism.
4. **Ease of Use**:


In [ ]:
print(ask_llm("What is deep learning?"))

Deep learning is a subset of machine learning that involves training artificial neural networks to perform tasks such as image recognition, speech recognition, and natural language processing. The goal of deep learning is to enable machines to learn from data and make predictions or decisions without being explicitly programmed.
In simple terms, deep learning uses multiple layers of neurons in the network, each with its own set of weights and biases. These weights and biases allow the network to "learn" patterns in the input data and then use those learned patterns to make predictions or classifications on new data.
One of the key features of deep learning is that it can automatically discover complex patterns in large datasets that may not be apparent to humans. This makes it particularly useful for tasks like image and speech recognition, where


In [ ]:
conversation_history = []

In [ ]:
def chat_with_memory(question):

    conversation_history.append({
        "role": "user",
        "content": question
    })

    inputs_text = tokenizer.apply_chat_template(
        conversation_history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        inputs_text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    conversation_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

In [ ]:
print(chat_with_memory("My name is ChiraswyKJ."))

Hello! I'm ChiraswyKJ, your AI companion here to help with any questions or tasks you might have. How can I assist you today?


In [ ]:
print(chat_with_memory("What is my name?"))

Your name is ChiraswyKJ.


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving OOP_notes.pdf to OOP_notes.pdf


In [ ]:
import os

print("Files in Colab:")
print(os.listdir("/content"))

Files in Colab:
['.config', 'OOP_notes.pdf', 'sample_data']


In [ ]:
pdf_filename = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_filename)

Uploaded PDF: OOP_notes.pdf


In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_filename)

print("Number of pages:", len(reader.pages))

Number of pages: 55


In [ ]:
text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        text += page_text + "\n"

print(text[:3000])

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

print("Uploaded:", pdf_filename)

Saving OOP_notes.pdf to OOP_notes (1).pdf
Uploaded: OOP_notes (1).pdf


In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_filename)

documents = []

for page_number, page in enumerate(reader.pages):
    text = page.extract_text()

    if text:
        documents.append({
            "text": text,
            "page": page_number + 1
        })

print("Pages extracted:", len(documents))

Pages extracted: 0


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = []

for document in documents:
    split_texts = splitter.split_text(document["text"])

    for chunk in split_texts:
        chunks.append({
            "text": chunk,
            "page": document["page"]
        })

print("Total chunks:", len(chunks))

Total chunks: 0


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [ ]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches: 0it [00:00, ?it/s]

Embedding shape: (0,)


In [ ]:
import numpy as np
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print("FAISS database created!")
print("Stored vectors:", index.ntotal)

IndexError: tuple index out of range

In [ ]:
print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(0,)


In [ ]:
print("Number of documents:", len(documents))
print("Number of chunks:", len(chunks))

Number of documents: 0
Number of chunks: 0


In [ ]:
if len(documents) > 0:
    print(documents[0]["text"][:1000])
else:
    print("No text was extracted from the PDF.")

No text was extracted from the PDF.


In [ ]:
!apt-get -qq install -y tesseract-ocr
!pip install -q pytesseract pdf2image

In [ ]:
import pytesseract
from pdf2image import convert_from_path

print("OCR tools ready!")

OCR tools ready!


In [ ]:
pages = convert_from_path(pdf_filename, dpi=200)

print("Pages converted:", len(pages))

PDFInfoNotInstalledError: Unable to get page count. Is poppler installed and in PATH?

In [ ]:
!apt-get -qq update
!apt-get -qq install -y poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up poppler-utils (22.02.0-2ubuntu0.13) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
pages = convert_from_path(pdf_filename, dpi=200)

print("Pages converted:", len(pages))

Pages converted: 55


In [ ]:
documents = []

for page_number, page_image in enumerate(pages):

    text = pytesseract.image_to_string(page_image)

    if text.strip():
        documents.append({
            "text": text,
            "page": page_number + 1
        })

print("Pages with extracted text:", len(documents))

Pages with extracted text: 55


In [ ]:
if documents:
    print(documents[0]["text"][:2000])
else:
    print("OCR could not extract any text.")

OOF

4 Sy, ;

Object OvfenkoL i eC

L, Entity IP Concer nod
har CX Ro fF Lokes Hol

in Real -
Woxlal LC tt, C#, Java, JavaSonriat , Py tren, .ek|
||

VY

Date + Code
-Atefoules  * Proceduses
‘Propeeles © Methods

 

 

Proceauve OP VE object OP
@ Padgram Ps hv Pdool Q) Proaram fs aPuPced

ete * funchfons” oT

n Prto Okects"
@© Inhevfence (x) © Tuhwe tence cy)
® “TOP - DOWN App soach @ Bottom UP APP SdaCl,
@ Virtua funchenc) ¢@ virtual function ci) |
® No code Reusa bl lity © ng Code can be
. Uatal .

Less Secure
© (NO cake hrcling ) © Encapsulahen fs Catof
@) Easy +0 add A te CS eo | hol nbd

E SO Fo he D Baty add new dalary
© gi ©, FORTAN, PASCAL > fa : Ctt, Java_,Pythe, |

 



In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = []

for document in documents:
    split_texts = splitter.split_text(document["text"])

    for chunk in split_texts:
        chunks.append({
            "text": chunk,
            "page": document["page"]
        })

print("Total chunks:", len(chunks))

Total chunks: 55


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [ ]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (55, 384)


In [ ]:
import numpy as np
import faiss

embeddings = np.asarray(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS database created!")
print("Vectors stored:", index.ntotal)

FAISS database created!
Vectors stored: 55


In [ ]:
def retrieve_documents(question, k=3):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(
        question_embedding,
        k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks[idx])

    return results

In [ ]:
results = retrieve_documents(
    "What is this document about?"
)

for result in results:
    print("PAGE:", result["page"])
    print(result["text"][:500])
    print("-" * 50)

PAGE: 38
© Pointers
x M eM a“ Manage mort dys Lain Cc

[ \ ( 7 J memory aALlocahen

2 * | malloc C ) ee Calfoc C ~¥
| freeo]
cloqlocatt mem bleep -

 

Ls5¢
2 CTE Tne & delele |

OReck Can be creaktd with

new “ Le Ais teo yeh Ch He
Aulete’

4 plrvay = new calakype,
mt *phr es pew Pb
mt Kptr = new intl to]

\

<q delle ptrvar
olelehe ptr
ark te [10] pir
--------------------------------------------------
PAGE: 41
* Okyeck & polntie
Class A
t ink x j
ve"

vo Pd put C irr a)
{ x=a4

void Usplay C)
t Gale x 4}

= A ai
A * ptr= Qa

® Access \ nq, Mem bees UAI Ne dot ¢ )
as-pul(s )
a1: oUselaul).

@ Recess te MeMb eed Uke Por nrbere.

ptr— pub (5 )
pm ~ displayO

@ ole referencing C*) & GC) operatde

(+ pty). purls)
(* pty) Usplay C)
--------------------------------------------------
PAGE: 17
* Classes & Obyecks

> Class classnamt

{

Access spedibee .
Varfable cloclaxah ons ;
ACCE&S Sed he.

F

Access Contes |

|. Acce cs

‘cess C's (data Relig )

funcHen du da ralens

Private Cdefautt)
Pub

In [ ]:
def answer_from_document(question):

    # 1. Retrieve relevant chunks
    results = retrieve_documents(question, k=3)

    # 2. Combine the retrieved text
    context = "\n\n".join(
        [
            f"[Page {r['page']}]\n{r['text']}"
            for r in results
        ]
    )

    # 3. Create the prompt
    prompt = f"""
You are an intelligent document assistant.

Answer the user's question using ONLY the information
provided in the document context below.

If the answer cannot be found in the context,
say:
"I could not find this information in the document."

DOCUMENT CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    # 4. Send prompt to LLM
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # 5. Generate answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

    # 6. Convert output to text
    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return answer, results

In [ ]:
answer, sources = answer_from_document(
    "What is this document about?"
)

print(answer)

This document is about pointers and their usage in programming languages. It explains how pointers work and provides examples of different types of pointers such as manual pointers, dynamic allocation, and deallocation. The document also covers the concept of classes and objects, including their properties and methods. Additionally, it discusses garbage collection and how pointers can be used for accessing memory locations. Overall, the document aims to provide a comprehensive overview of pointers and their applications in computer programming.


In [ ]:
print("\nSources used:")

for source in sources:
    print("Page:", source["page"])


Sources used:
Page: 38
Page: 41
Page: 17


In [ ]:
def chatbot(message, history):

    answer, sources = answer_from_document(message)

    pages = sorted(
        set(source["page"] for source in sources)
    )

    source_text = "\n\n**Sources:** " + ", ".join(
        f"Page {page}" for page in pages
    )

    return answer + source_text

In [ ]:
demo = gr.ChatInterface(
    fn=chatbot,
    title="🤖 Intelligent Document Chatbot",
    description="Ask questions about your uploaded PDF."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://23cc22515ccbd645a9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
